In [2]:
import os
import cv2

# folder_positif = 'positive' 
# folder_negatif = 'negative'

# # 1. Generate bg.txt (Untuk data negatif)
# with open('bg.txt', 'w') as f:
#     for filename in os.listdir(folder_negatif):
#         if filename.endswith(('.jpg', '.png', '.jpeg')):
#             f.write(f"{folder_negatif}/{filename}\n")
# print("File bg.txt berhasil dibuat!")

# # 2. Generate positives.info (Untuk data positif)
# with open('positives.info', 'w') as f:
#     for filename in os.listdir(folder_positif):
#         if filename.endswith(('.jpg', '.png', '.jpeg')):
#             filepath = f"{folder_positif}/{filename}"
#             # Karena semua gambar positif sudah di-resize ke 24x24 di script sebelumnya,
#             # kita bisa langsung tulis ukurannya secara statis
#             f.write(f"{filepath} 1 0 0 24 24\n")
print("File positives.info berhasil dibuat!")

File positives.info berhasil dibuat!


In [7]:
import time

# 1. Load DUA model sekaligus
# Model pendeteksi wajah bawaan OpenCV (Sangat akurat)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Model pendeteksi ngantuk buatanmu
cascade_path = './classifier/cascade.xml' 
yawn_cascade = cv2.CascadeClassifier(cascade_path)

cap = cv2.VideoCapture(0)
prev_time = 0

print("Kamera menyala! Tekan 'q' pada keyboard untuk keluar.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    start_det_time = time.time()

    # 2. LANGKAH PERTAMA: Cari Wajah dulu
    faces = face_cascade.detectMultiScale(
        gray, 
        scaleFactor=1.1,      # Diubah ke 1.1 agar OpenCV memindai lebih teliti
        minNeighbors=5, 
        minSize=(200, 200)    # Wajibkan ukuran wajah minimal cukup besar agar kotak kecil diabaikan
    )

    for (x, y, w, h) in faces:
        # (Opsional) Gambar kotak biru untuk area wajah agar kamu tahu batasnya
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        # 3. KUNCI AREA (ROI): Potong setengah wajah bagian bawah untuk mencari mulut
        # Ini mencegah mata dikira mulut yang sedang menguap
        setengah_bawah_y = y + int(h/2)
        roi_gray = gray[setengah_bawah_y:y+h, x:x+w]
        
        # 4. LANGKAH KEDUA: Deteksi ngantuk HANYA di area 'roi_gray' (setengah wajah bawah)
        yawns, rejectLevels, levelWeights = yawn_cascade.detectMultiScale3(
            roi_gray, 
            scaleFactor=1.2,       
            minNeighbors=3,       
            minSize=(30, 30),      # Bisa dikecilkan lagi karena area pencariannya sudah sempit
            outputRejectLevels=True 
        )

        if len(yawns) > 0:
            for (mx, my, mw, mh), weight in zip(yawns, levelWeights):
                confidence_score = round(float(weight), 2)
                
                # Filter skor (sesuaikan dengan seleramu)
                if confidence_score > 0.8: 
                    label = f'Ngantuk! Skor: {confidence_score}'
                    
                    # Hitung koordinat asli kotak merah di layar (karena tadi mx, my diukur dari dalam kotak ROI)
                    abs_x = x + mx
                    abs_y = setengah_bawah_y + my
                    
                    cv2.rectangle(frame, (abs_x, abs_y), (abs_x+mw, abs_y+mh), (0, 0, 255), 3)
                    cv2.putText(frame, label, (abs_x, abs_y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    # Menghitung Waktu & FPS
    end_det_time = time.time()
    det_time_ms = (end_det_time - start_det_time) * 1000 
    curr_time = time.time()
    fps = 1 / (curr_time - prev_time) if (curr_time - prev_time) > 0 else 0
    prev_time = curr_time

    cv2.putText(frame, f'FPS: {int(fps)}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(frame, f'Deteksi: {int(det_time_ms)} ms', (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cv2.imshow('Uji Coba Haar Cascade (Dengan ROI)', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Kamera menyala! Tekan 'q' pada keyboard untuk keluar.
